# Memory and Message History

**What you will build:** multi-turn conversations in PydanticAI. The whole `WindowMemory`/`SummaryMemory` machinery you wrote in 0.6 becomes a single argument — but the underlying trade-offs don't disappear, and it's worth seeing why.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/14_memory_and_history.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.2 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## `message_history`: memory in one argument

Every run returns its messages via `result.all_messages()`. Feed them into the next run with `message_history=` and the agent remembers. That's the same append-the-list mechanic from 0.1, now handled for you.

In [ ]:
agent = Agent(model, system_prompt="You are a concise assistant.")

r1 = await agent.run("My name is Sam and I love jazz.")
print(r1.output)

r2 = await agent.run("What's my name, and what music do I like?",
                     message_history=r1.all_messages())
print(r2.output)

## A reusable chat loop

Wrap it so each turn carries the growing history forward automatically.

In [ ]:
agent = Agent(model, system_prompt="You are a friendly assistant with a good memory.")
history = []

async def chat(user_text):
    result = await agent.run(user_text, message_history=history)
    history[:] = result.all_messages()      # carry everything forward
    return result.output

print(await chat("I'm planning a trip to Japan in April."))
print(await chat("I especially love food and temples."))
print(await chat("Given all that, suggest one city for me."))   # uses both earlier turns

```{note}
PydanticAI makes memory convenient, not free. `history` still grows every turn, and it still has to fit the context window (0.0, 0.6). For long chats you apply the *same* strategies you built by hand — trim or summarize `history` before passing it back. Frameworks automate the plumbing, not the budgeting.
```

::::{dropdown} 🔍 What is inside `all_messages()`
:color: info

It's the structured record of the run: the system prompt, each user request, the model's responses, and any tool calls and their results — the typed version of the `messages` list you built by hand in Block 0. Print `len(history)` after a few turns to watch it grow; that growth is exactly what LangGraph's **checkpointer** (2.3) will persist for you across sessions.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Prove it's the memory.** Remove `message_history=history` from `chat` and re-run the trip conversation. It forgets, exactly like the statelessness demo in 0.1.
2. **Bound it.** Keep only the last 6 messages: `history[:] = result.all_messages()[-6:]`. Watch older details drop — you've reimplemented `WindowMemory` from 0.6 in one line.
::::

**What's next:** in **1.5** we add **guardrails** — validating what goes in and what comes out, and defending against prompt injection.